In [1]:
# ============================================================
# CELL 1 — Load Odia streaming k2 model + fetch tokens file
# ============================================================
import torch
import os
import urllib.request

# --- adjust this to your Kaggle input path for the Odia .pt ---
PT_PATH     = "/kaggle/input/models/nishargnargund/ori-asr-odia/pytorch/default/1/SPRING_INX_streaming_k2_Odia.pt"

# Tokens file (download to working dir)
TOKENS_URL  = "https://asr.iitm.ac.in/SPRING_INX/models/tokens_list/SPRING_INX_Odia_tokens.txt"
TOKENS_PATH = "/kaggle/working/SPRING_INX_Odia_tokens.txt"

# --- Download tokens file ---
if not os.path.exists(TOKENS_PATH):
    print(f"Downloading tokens: {TOKENS_URL}")
    urllib.request.urlretrieve(TOKENS_URL, TOKENS_PATH)
print(f"✅ Tokens file: {TOKENS_PATH}  ({os.path.getsize(TOKENS_PATH)} bytes)")

# --- Load model as TorchScript (jit-scripted AsrModel, NOT a state dict) ---
model = torch.jit.load(PT_PATH, map_location="cpu")
model.eval()
print("\n✅ Model loaded")
print(f"Type : {type(model)}")
print(f"Class: {getattr(model, 'original_name', 'N/A')}")

# --- Top-level submodules — expect the same 6 as Gujarati ---
print("\n=== Submodules ===")
for name, mod in model.named_children():
    print(f"  {name:18s} -> {getattr(mod, 'original_name', type(mod).__name__)}")

# --- Vocab from tokens file (WILL differ from Gujarati's 900 — never assume) ---
with open(TOKENS_PATH, encoding="utf-8") as f:
    tokens = [line.strip().split()[0] for line in f if line.strip()]
print(f"\n=== Vocab ===")
print(f"  vocab_size (from tokens file) = {len(tokens)}")
print(f"  first 5 tokens : {tokens[:5]}")
print(f"  last 5 tokens  : {tokens[-5:]}")

# --- Check if tokens are also baked into the .pt (may skip the download entirely) ---
print("\n=== Tokens embedded in .pt? ===")
for attr in ["tokens", "token_table", "sp_model", "bpe_model", "vocab"]:
    if hasattr(model, attr):
        print(f"  found: model.{attr}")

✅ Tokens file: /kaggle/working/SPRING_INX_Odia_tokens.txt  (2723 bytes)

✅ Model loaded
Type : <class 'torch.jit._script.RecursiveScriptModule'>
Class: AsrModel

=== Submodules ===
  encoder_embed      -> Conv2dSubsampling
  encoder            -> StreamingEncoderModel
  decoder            -> Decoder
  joiner             -> Joiner
  simple_am_proj     -> Linear
  simple_lm_proj     -> Linear

=== Vocab ===
  vocab_size (from tokens file) = 303
  first 5 tokens : ['<blk>', '<sos/eos>', '<unk>', '▁', 'ା']
  last 5 tokens  : ['अ', 'ଙ', '#0', '#1', '#2']

=== Tokens embedded in .pt? ===


In [2]:
# ============================================================
# CELL 2 — Derive architecture constants from the Odia model
# ============================================================
import torch

BATCH_SIZE = 1
device = torch.device("cpu")

# --- Encoder streaming constants ---
chunk_size       = int(model.encoder.chunk_size)
left_context_len = int(model.encoder.left_context_len)
context_size     = int(model.decoder.context_size)

print("=== Encoder / Decoder constants ===")
print(f"  chunk_size        = {chunk_size}")
print(f"  left_context_len  = {left_context_len}")
print(f"  context_size      = {context_size}")

# --- encoder_embed geometry ---
for attr in ["out_width", "layer3_channels"]:
    try:
        print(f"  encoder_embed.{attr} = {getattr(model.encoder_embed, attr)}")
    except Exception as e:
        print(f"  encoder_embed.{attr} -> N/A ({e})")

# --- Init states: count + shapes (this defines the tracing contract) ---
states = model.encoder.get_init_states(batch_size=BATCH_SIZE, device=device)
print(f"\n=== Init states ===")
print(f"  total states = {len(states)}")
print(f"  states[-2] (embed cache)   : {states[-2].shape} {states[-2].dtype}")
print(f"  states[-1] (processed_lens) : {states[-1].shape} {states[-1].dtype}")

# --- Derive raw-frame chunk length: chunk_size*2 + 13 (streaming subsample formula) ---
RAW_FRAMES = chunk_size * 2 + 13
FEAT_DIM   = 80
print(f"\n=== Derived input geometry ===")
print(f"  FEAT_DIM     = {FEAT_DIM}")
print(f"  RAW_FRAMES   = {RAW_FRAMES}   (= chunk_size*2 + 13)")

# --- Stash globals for later cells ---
CHUNK_SIZE       = chunk_size
LEFT_CONTEXT_LEN = left_context_len
N_STATES         = len(states)
VOCAB_SIZE       = len(tokens)
print(f"\n✅ Constants ready: CHUNK_SIZE={CHUNK_SIZE}, LEFT_CONTEXT_LEN={LEFT_CONTEXT_LEN}, "
      f"N_STATES={N_STATES}, VOCAB_SIZE={VOCAB_SIZE}")

=== Encoder / Decoder constants ===
  chunk_size        = 32
  left_context_len  = 128
  context_size      = 2
  encoder_embed.out_width = 19
  encoder_embed.layer3_channels = 128

=== Init states ===
  total states = 98
  states[-2] (embed cache)   : torch.Size([1, 128, 3, 19]) torch.float32
  states[-1] (processed_lens) : torch.Size([1]) torch.int32

=== Derived input geometry ===
  FEAT_DIM     = 80
  RAW_FRAMES   = 77   (= chunk_size*2 + 13)

✅ Constants ready: CHUNK_SIZE=32, LEFT_CONTEXT_LEN=128, N_STATES=98, VOCAB_SIZE=303


In [4]:
# --- Vocab reconciliation (model is authoritative) ---
MODEL_VOCAB = joiner_out.shape[-1]          # 300 — real emission classes
TOKENS_VOCAB = VOCAB_SIZE                     # 303 — includes #0/#1/#2 disambig symbols

print(f"\n  model vocab (joiner out) = {MODEL_VOCAB}")
print(f"  tokens file entries      = {TOKENS_VOCAB}")

# The tokens file appends k2 disambiguation symbols (#0,#1,#2) not emitted by the AM.
extra = TOKENS_VOCAB - MODEL_VOCAB
tail = tokens[MODEL_VOCAB:] if extra > 0 else []
print(f"  extra token entries ({extra}): {tail}")

assert extra >= 0, f"tokens file ({TOKENS_VOCAB}) smaller than model vocab ({MODEL_VOCAB}) — real mismatch!"
assert all(t.startswith('#') for t in tail), \
    f"Unexpected trailing tokens (not #-disambig): {tail} — investigate before proceeding"

# Lock in the authoritative value for later cells
VOCAB_SIZE = MODEL_VOCAB
print(f"\n✅ Reconciled. VOCAB_SIZE set to model value = {VOCAB_SIZE}")
print(f"   (tokens[{MODEL_VOCAB}:] = {tail} are disambig symbols, excluded at transcribe time)")
print("✅ ALL FORWARD PASSES SUCCESSFUL — safe to trace")


  model vocab (joiner out) = 300
  tokens file entries      = 303
  extra token entries (3): ['#0', '#1', '#2']

✅ Reconciled. VOCAB_SIZE set to model value = 300
   (tokens[300:] = ['#0', '#1', '#2'] are disambig symbols, excluded at transcribe time)
✅ ALL FORWARD PASSES SUCCESSFUL — safe to trace


In [5]:
# ============================================================
# CELL 4 — Wrap + trace the ENCODER
# ============================================================
import torch
import torch.nn as nn

class EncoderTraceWrapper(nn.Module):
    """Registers encoder as submodule; flattens the 98-state list to *states
    so the tracer can handle it (list-indexing inside forward isn't traceable)."""
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder   # registers as submodule ✅

    def forward(self, x, x_lens, *states):
        states_list = list(states)
        enc_out, enc_lens, new_states = self.encoder.forward(x, x_lens, states_list)
        return (enc_out, enc_lens) + tuple(new_states)

# --- Dummy inputs (use derived constants, not hardcoded) ---
states     = model.encoder.get_init_states(batch_size=1, device=torch.device("cpu"))
x_dummy    = torch.randn(1, RAW_FRAMES, FEAT_DIM)
lens_dummy = torch.tensor([RAW_FRAMES], dtype=torch.int32)

wrapper = EncoderTraceWrapper(model.encoder)
wrapper.eval()

# --- Verify wrapper forward matches the original encoder ---
with torch.no_grad():
    ref_out, ref_lens, ref_states = model.encoder.forward(x_dummy, lens_dummy, states)
    outs = wrapper(x_dummy, lens_dummy, *states)

wrap_diff = (outs[0] - ref_out).abs().max().item()
print(f"Wrapper forward: enc_out={outs[0].shape}, n_states_out={len(outs)-2}")
print(f"Wrapper vs original: max_diff={wrap_diff:.2e}  {'✅ pass' if wrap_diff < 1e-4 else '❌ FAIL'}")
assert wrap_diff < 1e-4
assert len(outs) - 2 == N_STATES, f"state count {len(outs)-2} != {N_STATES}"

# --- Trace ---
dummy_args = (x_dummy, lens_dummy) + tuple(states)
print("\nTracing encoder ...")
with torch.no_grad():
    traced_encoder = torch.jit.trace(
        wrapper, dummy_args, strict=False, check_trace=False
    )
    outs_t = traced_encoder(x_dummy, lens_dummy, *states)

trace_diff = (outs_t[0] - ref_out).abs().max().item()
print(f"Traced vs original: max_diff={trace_diff:.2e}  {'✅ pass' if trace_diff < 1e-4 else '❌ FAIL'}")
assert trace_diff < 1e-4
print("✅ Encoder traced successfully")

Wrapper forward: enc_out=torch.Size([1, 16, 512]), n_states_out=98
Wrapper vs original: max_diff=0.00e+00  ✅ pass

Tracing encoder ...
Traced vs original: max_diff=0.00e+00  ✅ pass
✅ Encoder traced successfully


In [6]:
# ============================================================
# CELL 5 — Trace the DECODER and JOINER
# ============================================================
import torch

# ---------------- DECODER ----------------
# Signature: forward(y, need_pad). Trace with need_pad=False (inference mode).
y_dummy = torch.zeros(1, context_size, dtype=torch.int64)   # [1, 2]

print("Tracing decoder ...")
with torch.no_grad():
    ref_dec = model.decoder.forward(y_dummy, need_pad=False)
    traced_decoder = torch.jit.trace(
        model.decoder,
        (y_dummy, torch.tensor(False)),
        check_trace=False,
    )
    dec_t = traced_decoder(y_dummy, torch.tensor(False))

dec_diff = (dec_t - ref_dec).abs().max().item()
print(f"  decoder out: {dec_t.shape}")
print(f"  Traced vs original: max_diff={dec_diff:.2e}  {'✅ pass' if dec_diff < 1e-4 else '❌ FAIL'}")
assert dec_diff < 1e-4

# ---------------- JOINER ----------------
# Signature: forward(enc, dec, project_input). At inference project_input=True.
# Shapes per-frame at decode time: enc [1,1,1,512], dec [1,1,1,512]
enc_dummy = torch.randn(1, 1, 1, 512)
dec_dummy = torch.randn(1, 1, 1, 512)

print("\nTracing joiner ...")
with torch.no_grad():
    ref_join = model.joiner.forward(enc_dummy, dec_dummy, project_input=True)
    traced_joiner = torch.jit.trace(
        model.joiner,
        (enc_dummy, dec_dummy, torch.tensor(True)),
        check_trace=False,
    )
    join_t = traced_joiner(enc_dummy, dec_dummy, torch.tensor(True))

join_diff = (join_t - ref_join).abs().max().item()
print(f"  joiner out: {join_t.shape}   <- last dim = {join_t.shape[-1]} (model vocab)")
print(f"  Traced vs original: max_diff={join_diff:.2e}  {'✅ pass' if join_diff < 1e-4 else '❌ FAIL'}")
assert join_diff < 1e-4
assert join_t.shape[-1] == VOCAB_SIZE, f"joiner vocab {join_t.shape[-1]} != {VOCAB_SIZE}"

print("\n✅ Decoder and joiner traced successfully")
print("   traced_encoder / traced_decoder / traced_joiner all ready for mobile export")

Tracing decoder ...
  decoder out: torch.Size([1, 1, 512])
  Traced vs original: max_diff=0.00e+00  ✅ pass

Tracing joiner ...
  joiner out: torch.Size([1, 1, 1, 300])   <- last dim = 300 (model vocab)
  Traced vs original: max_diff=0.00e+00  ✅ pass

✅ Decoder and joiner traced successfully
   traced_encoder / traced_decoder / traced_joiner all ready for mobile export


/usr/local/lib/python3.12/dist-packages/torch/jit/_trace.py:1016: UserWarning: The input to trace is already a ScriptModule, tracing it is a no-op. Returning the object as is.
  traced_func = _trace_impl(


In [7]:
# ============================================================
# CELL 6 — Optimize for mobile + export 3 .ptl files
# ============================================================
import torch
import os
from torch.utils.mobile_optimizer import optimize_for_mobile

OUT_DIR = "/kaggle/working"

traced_models = {
    "encoder": traced_encoder,
    "decoder": traced_decoder,
    "joiner":  traced_joiner,
}

print("🚀 Compiling for PyTorch Mobile (Lite Interpreter)...\n")

ptl_paths = {}
for name, tm in traced_models.items():
    ptl_path = os.path.join(OUT_DIR, f"{name}.ptl")
    try:
        optimized = optimize_for_mobile(tm)
        optimized._save_for_lite_interpreter(ptl_path)
        size_mb = os.path.getsize(ptl_path) / 1e6
        ptl_paths[name] = ptl_path
        print(f"✅ {name}.ptl  ({size_mb:.1f} MB)")
    except Exception as e:
        print(f"❌ {name}: {e}")

print(f"\n✅ Exported {len(ptl_paths)}/3 .ptl files to {OUT_DIR}")

# --- Also save init states so the phone never needs the original .pt ---
init_states = model.encoder.get_init_states(batch_size=1, device=torch.device("cpu"))
torch.save(init_states, os.path.join(OUT_DIR, "init_states.pt"))
print(f"✅ init_states.pt saved ({len(init_states)} tensors)")

🚀 Compiling for PyTorch Mobile (Lite Interpreter)...

✅ encoder.ptl  (261.1 MB)
✅ decoder.ptl  (0.6 MB)
✅ joiner.ptl  (2.7 MB)

✅ Exported 3/3 .ptl files to /kaggle/working
✅ init_states.pt saved (98 tensors)


/tmp/ipykernel_136/1955876345.py:23: DeprecationWarning: Lite Interpreter is deprecated. Please consider switching to ExecuTorch.             https://docs.pytorch.org/executorch/stable/getting-started.html
  optimized._save_for_lite_interpreter(ptl_path)


In [11]:
!pip install -q transformers
import torch
from transformers import VitsModel, AutoTokenizer
import torchaudio

# MMS-TTS Odia
tts = VitsModel.from_pretrained("facebook/mms-tts-ory")
tok = AutoTokenizer.from_pretrained("facebook/mms-tts-ory")

text_in = "ଆପଣ କେମିତି ଅଛନ୍ତି"
inputs = tok(text_in, return_tensors="pt")
with torch.no_grad():
    out = tts(**inputs).waveform   # [1, T] at model sample rate

sr_tts = tts.config.sampling_rate
wav16 = torchaudio.functional.resample(out, sr_tts, 16000)
torchaudio.save("/kaggle/working/odia_test.wav", wav16, 16000)
print(f"✅ /kaggle/working/odia_test.wav  (from {sr_tts}Hz → 16kHz)")
print(f"   input text: {text_in}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 7.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.4.2 which is incompatible.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/954 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

✅ /kaggle/working/odia_test.wav  (from 16000Hz → 16kHz)
   input text: ଆପଣ କେମିତି ଅଛନ୍ତି


In [12]:
# ============================================================
# CELL 7 — Reload .ptl from disk + full transcription test
# ============================================================
import torch
import torchaudio
import torchaudio.compliance.kaldi as kaldi

OUT_DIR = "/kaggle/working"

# --- Reload the .ptl files fresh (proves they work standalone) ---
enc_ptl = torch.jit.load(f"{OUT_DIR}/encoder.ptl", map_location="cpu"); enc_ptl.eval()
dec_ptl = torch.jit.load(f"{OUT_DIR}/decoder.ptl", map_location="cpu"); dec_ptl.eval()
join_ptl = torch.jit.load(f"{OUT_DIR}/joiner.ptl", map_location="cpu"); join_ptl.eval()
print("✅ All 3 .ptl reloaded from disk")

# --- Tokens (Odia uses <blk>, not <blank> — note the exclusion list) ---
with open(TOKENS_PATH, encoding="utf-8") as f:
    tokens = [line.strip().split()[0] for line in f if line.strip()]
EXCLUDE = {"<blk>", "<eps>", "<blank>", "<unk>", "<sos/eos>", "#0", "#1", "#2"}

def transcribe(audio_path):
    wav, sr = torchaudio.load(audio_path)
    if sr != 16000:
        wav = torchaudio.functional.resample(wav, sr, 16000)
    if wav.shape[0] > 1:
        wav = wav.mean(0, keepdim=True)

    # Kaldi fbank, 80 mel, NO CMVN (matches training)
    mel = kaldi.fbank(
        wav, num_mel_bins=FEAT_DIM,
        frame_length=25.0, frame_shift=10.0,
        high_freq=8000, low_freq=20,
        sample_frequency=16000, use_energy=False,
    )  # [T, 80]

    # Pad to a multiple of RAW_FRAMES (77) and chunk
    T = mel.shape[0]
    pad = (-T) % RAW_FRAMES
    if pad:
        mel = torch.cat([mel, torch.zeros(pad, FEAT_DIM)], dim=0)
    chunks = mel.reshape(-1, RAW_FRAMES, FEAT_DIM)
    print(f"  Audio: {wav.shape[1]/16000:.2f}s → {T} frames → {len(chunks)} chunks")

    # Init states from original .pt (at deploy time: load init_states.pt instead)
    states = model.encoder.get_init_states(batch_size=1)
    hyp = [0, 0]   # context_size=2, blank=0

    with torch.no_grad():
        for chunk in chunks:
            x = chunk.unsqueeze(0)
            x_lens = torch.tensor([RAW_FRAMES], dtype=torch.int32)

            enc_tuple = enc_ptl(x, x_lens, *states)
            enc_out = enc_tuple[0]
            states = list(enc_tuple[2:])

            for t in range(enc_out.shape[1]):
                y = torch.tensor([[hyp[-2], hyp[-1]]], dtype=torch.int64)
                dec_out = dec_ptl(y, torch.tensor(False))     # need_pad=False
                enc_frame = enc_out[:, t, :]
                dec_frame = dec_out[:, 0, :]
                logits = join_ptl(enc_frame, dec_frame, True) # project_input=True
                token = logits[0].argmax().item()
                if token != 0:                                # 0 = blank
                    hyp.append(token)

    ids = hyp[2:]
    text = "".join(
        tokens[t].replace("▁", " ")
        for t in ids
        if 0 <= t < len(tokens) and tokens[t] not in EXCLUDE
    )
    return text.strip()

# --- Point this at your Odia test wav (16kHz mono) ---
WAV = "/kaggle/working/odia_test.wav"
print(f"\nTranscribing: {WAV}")
text = transcribe(WAV)
print(f"\n✅ Transcript: '{text}'")

✅ All 3 .ptl reloaded from disk

Transcribing: /kaggle/working/odia_test.wav
  Audio: 1.81s → 179 frames → 3 chunks

✅ Transcript: 'ଆପଣ ବି ଅଛନ୍ତି'


In [13]:
# ============================================================
# CELL 8 — Package Odia .ptl bundle for release
# ============================================================
import zipfile, os

OUT_DIR = "/kaggle/working"
zip_path = f"{OUT_DIR}/odia_asr_ptl.zip"
files = ["encoder.ptl", "decoder.ptl", "joiner.ptl",
         "init_states.pt", "SPRING_INX_Odia_tokens.txt"]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        p = os.path.join(OUT_DIR, f)
        if os.path.exists(p):
            z.write(p, arcname=f)
            print(f"  added: {f}  ({os.path.getsize(p)/1e6:.1f} MB)")
        else:
            print(f"  ⚠️ missing: {f}")

print(f"\n✅ Bundle: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)")

from IPython.display import FileLink
FileLink(zip_path)

  added: encoder.ptl  (261.1 MB)
  added: decoder.ptl  (0.6 MB)
  added: joiner.ptl  (2.7 MB)
  added: init_states.pt  (1.9 MB)
  added: SPRING_INX_Odia_tokens.txt  (0.0 MB)

✅ Bundle: /kaggle/working/odia_asr_ptl.zip  (244.5 MB)


/kaggle/working/odia_asr_ptl.zip

In [14]:
from IPython.display import FileLink, display
display(FileLink("odia_asr_ptl.zip"))   # relative path, run from /kaggle/working

/kaggle/working/odia_asr_ptl.zip